# PyTorch and Autodifferentiation

## Learning Objectives

- Identify the limitations of standard numerical libraries (NumPy/SciPy) for machine learning.
- Explain the mechanics of automatic differentiation (autodiff) and computational graphs.
- Differentiate between forward-pass and backward-pass autodiff.
- Create and manipulate PyTorch tensors, tracking their gradients.
- Build and train a neural network using PyTorch's torch.nn and torch.optim modules.
- Manage hardware acceleration by offloading computations to GPUs.

In [ ]:
import torch
import numpy as np

# Create a 1D tensor
x = torch.tensor([1.0, 2.0, 3.0])
print(x)
print(type(x))

In [ ]:
# Create a NumPy array
y = np.array([4.0, 5.0, 6.0])
print(y)
print(type(y))

# Convert NumPy array to PyTorch tensor
x = torch.from_numpy(y)
print(x)
print(type(x))

# Convert PyTorch tensor back to NumPy array
y = x.numpy()
print(y)
print(type(y))

In [ ]:
x = torch.tensor(2.0)
print(x)
print(x.shape)

## Tracking Derivatives With `requires_grad`

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
print(x)

The Autograd Engine in Action
Let's revisit our toy example from earlier:

$$y = sin(x)\times x^2$$

In [ ]:
# Define our function y = sin(x) * x^2 using torch operations
def f(x): 
    return torch.sin(x) * x**2

x = torch.tensor(1.234, requires_grad=True)
y = f(x)
print(f"y = {y.item()}")

# Do the backward pass through f(x)
y.backward()
print("dy/dx =", x.grad.item())

In [ ]:
# Must detach the PyTorch tensor before converting to a NumPy array if it is tracking gradients
x_val = x.detach().numpy()
analytic = 2 * x_val * np.sin(x_val) + (x_val**2) * np.cos(x_val)

print(f"autodiff : {x.grad}")
print(f"analytic : {analytic}")
print(f"difference: {abs(x.grad.numpy() - analytic)}")

## The Gradient Accumulation Trap

In [ ]:
# Redefine x and its gradient
x = torch.tensor(1.234, requires_grad=True)

# First backward pass (computes dy/dx)
f(x).backward()
print("after first backward:", x.grad)

# Second backward pass (computes dy/dx again and adds to existing gradient)
f(x).backward()
print("after second backward:", x.grad)

In [ ]:
# Clear existing gradient and compute again
x.grad.zero_()
f(x).backward()
print("after zero_ then backward:", x.grad.item())

## Optimization Using Autodiff

In [ ]:
# Target value for y (we want to find x such that f(x) = 1.0)
y_target = torch.tensor(1.0)

# Initial guess for x and learning rate
x = torch.tensor(2.5, requires_grad=True)
lr = 1E-3

# Optimization loop
for step in range(100):
    # Evaluate function and compute mean squared error loss
    y = f(x)
    loss = (y - y_target)**2
    loss.backward()

    # Disable gradient tracking for the manual update step
    with torch.no_grad():
        x -= lr * x.grad
        x.grad.zero_()

    if step % 5 == 0 or step == 99:
        print(f"step {step:02d}  x={x.item(): .4f}  y={y.item(): .4f}  log(loss)={np.log10(loss.item()): .4f}")

## The PyTorch Ecosystem: `torch.nn` and `torch.optim`

In [ ]:
import torch.nn as nn

# Define a simple model as a subclass of nn.Module
class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Define a trainable parameter x
        self.x = nn.Parameter(torch.tensor(2.5))

    # The forward method defines the computation performed at every call
    def forward(self):
        return torch.sin(self.x) * self.x**2

In [ ]:
# Initialize the model object and perform a forward pass
toymodel = ToyModel()
y = toymodel()
print(f"y = {y}")

# Access the model parameter x
print(f"model parameter x = {toymodel.x.item()}")

# Inspect the model parameters
for name, p in toymodel.named_parameters():
    print(f"parameter name: {name}, value: {p.item()}, requires_grad: {p.requires_grad}")

## Streamlined Training With `torch.optim`

In [ ]:
from torch import optim

y_target = torch.tensor(1.0)

# Use the SGD optimizer from torch.optim instead of manual gradient descent
toyoptimizer = optim.SGD(toymodel.parameters(), lr=1E-3)

# Optimization loop using the optimizer
for step in range(100):
    y = toymodel()
    loss = (y - y_target)**2

    loss.backward()
    toyoptimizer.step()
    toyoptimizer.zero_grad()

    if step % 5 == 0 or step == 99:
        print(f"step {step:02d}  x={toymodel.x.item(): .4f}  y={y.item(): .4f}  log(loss)={np.log10(loss.item()): .4f}")

## Hardware Acceleration: CPU Versus GPU

In [ ]:
# Check if CUDA is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.tensor(1.234, requires_grad=True)
print("x device:", x.device)

x = x.to(device)
print("x device after move:", x.device)

y = torch.sin(x) * x**2
y.backward()

print("dy/dx:", x.grad)

## Models and Devices
Models are just collections of tensors. Moving a model to a GPU moves all of its parameters.

In [ ]:
# Move a model to the selected device
toymodel = ToyModel()
toymodel = toymodel.to(device)

# Print the model parameters and their devices
for name, p in toymodel.named_parameters():
    print(f"parameter name: {name}, value: {p.item()}, requires_grad: {p.requires_grad}, device: {p.device}")

In [ ]:
# Perform a forward pass on the selected device
target = torch.tensor(1.0).to(device)
toyoptimizer = optim.SGD(toymodel.parameters(), lr=1E-3)

# Do one optimization step
toyoptimizer.zero_grad()
y = toymodel()
loss = (y - target)**2
loss.backward()
toyoptimizer.step()

print("x =", toymodel.x.item(), "loss =", loss.item())

## A First Application: Fitting a Parabola

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Define a random number seed for reproducibility
torch.manual_seed(0)

# Sample synthetic dataset from the function y = x^2 + noise
N = 2048
x = 2.0 * torch.rand(N, 1) - 1.0
y = x**2 + 0.03 * torch.randn(N, 1)

# Train/val split (80% for training and 20% for validation)
n_train = int(0.8 * N)
x_train, x_valid = x[:n_train], x[n_train:]
y_train, y_valid = y[:n_train], y[n_train:]

train_ds = TensorDataset(x_train, y_train)
val_ds = TensorDataset(x_valid, y_valid)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
valid_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

print("train batches:", len(train_loader), "val batches:", len(valid_loader))

In [ ]:
# Define the model: a tiny MLP
class ParabolaMLP(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.hidden = hidden
        
        self.net = nn.Sequential(
            nn.Linear(1, self.hidden),
            nn.ReLU(),
            nn.Linear(self.hidden, self.hidden),
            nn.ReLU(),
            nn.Linear(self.hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

mlp = ParabolaMLP(hidden=32).to(device)
print(mlp)

## Training the Model
We will define our loss function $L$ as the Mean Squared Error (MSE) and use the Adam optimizer with a learning rate $\eta$.

In [ ]:
loss_fn = nn.MSELoss()
mlpopt = optim.Adam(mlp.parameters(), lr=1E-3)

In [ ]:
# Make code modular by defining an evaluation function
def evaluate(model, loader):
    model.eval() # Change model to evaluation mode
    total_loss = 0.0
    n = 0
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)
            loss = loss_fn(pred, yb)
            
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
            
    return total_loss / n

epochs = 100

# Training loop
for epoch in range(1, epochs + 1):
    mlp.train() # Change model to training mode
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        ypred = mlp(xb)
        loss = loss_fn(ypred, yb)

        loss.backward()
        mlpopt.step()
        mlpopt.zero_grad()

    # Evaluate and print losses every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        train_loss = evaluate(mlp, train_loader)
        valid_loss = evaluate(mlp, valid_loader)
        print(f"epoch {epoch:03d} | train log(MSE)={np.log10(train_loss):.5f} | val log(MSE)={np.log10(valid_loss):.5f}")

## Evaluating the Results

In [ ]:
import matplotlib.pyplot as plt

# Put model in eval mode
mlp.eval()

# Generate a smooth x grid
x_plot = torch.linspace(-1.0, 1.0, 400).unsqueeze(1).to(device)

with torch.no_grad():
    y_pred = mlp(x_plot).cpu()

# Move to CPU for plotting
x_plot = x_plot.cpu()
y_true = x_plot**2

fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')

# Training data (subsample for clarity)
ax.scatter(
    x_train[::10].numpy(),
    y_train[::10].numpy(),
    s=10,
    alpha=0.4,
    label="training data",
)

# True function
ax.plot(
    x_plot.numpy(),
    y_true.numpy(),
    linewidth=2,
    label="true y = x²",
)

# Model prediction
ax.plot(
    x_plot.numpy(),
    y_pred.numpy(),
    linewidth=2,
    linestyle="--",
    label="MLP prediction",
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## Try out for Gudel Benz relation and find a correlation fit